# Model 2 — Complex CNN — class-weighted experiment

Tuần tự: các khối Conv2D, BatchNormalization, ReLU và MaxPooling2D được xếp nối tiếp.

**Thử nghiệm cân bằng lớp:** dùng class weights tính từ tập train, lịch học có giảm learning rate khi validation loss chững và giới hạn epoch cao hơn. Đây là cấu hình cần kiểm chứng bằng kết quả raw, không mặc định là tốt hơn.

## 1. Cấu hình và thư viện

Khối này nạp TensorFlow, NumPy và các hàm đánh giá; đặt seed để kết quả có tính lặp lại tốt hơn; chọn `cut` hoặc `raw`; tìm thư mục gốc và chuẩn bị nơi lưu kết quả. Hãy dùng cùng một dataset và seed khi đối chiếu các kiến trúc.

Trong notebook thử nghiệm cân bằng, trọng số lớp được tính riêng từ số ảnh của tập train; không dùng validation/test để tính trọng số.

In [1]:
import csv
import json
import os
import time
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras import layers, models

physical_gpus = tf.config.list_physical_devices("GPU")
if not physical_gpus:
    raise RuntimeError(
        "TensorFlow did not detect a GPU. Select a CUDA-enabled TensorFlow environment "
        "before running this notebook."
    )

for gpu in physical_gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

print("TensorFlow GPUs:", physical_gpus)
with tf.device("/GPU:0"):
    gpu_probe = tf.linalg.matmul(tf.ones((64, 64)), tf.ones((64, 64)))
print("GPU compute check:", gpu_probe.device)
if "GPU:0" not in gpu_probe.device:
    raise RuntimeError(f"TensorFlow did not execute the GPU check on GPU: {gpu_probe.device}")

del gpu_probe


2026-10-10 00:00:24.862880: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-10-10 00:00:24.872009: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-10 00:00:24.872026: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
/home/pc/miniconda3/envs/pytorch_gpu/lib/python3.11/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


TensorFlow GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU compute check: /job:localhost/replica:0/task:0/device:GPU:0


In [2]:
DATASET = "raw"
MODEL_NAME = "complex_cnn_balanced"
SEED = 42
tf.keras.utils.set_random_seed(SEED)

# Find the project root even when this notebook is opened from a nested folder.
ROOT = Path.cwd().resolve()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "data" / DATASET).is_dir():
        ROOT = candidate
        break

data_dir = ROOT / "data" / DATASET
if not data_dir.is_dir():
    raise FileNotFoundError(f"Cannot find dataset folder: {data_dir}")

OUTPUT_DIR = ROOT / "src" / "Model_2" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")
print(f"Dataset: {data_dir}")

Project root: /home/pc/AAA/project/deepnn/G8_AI66A_DNN
Dataset: /home/pc/AAA/project/deepnn/G8_AI66A_DNN/data/raw


## 2. Xây dựng kiến trúc CNN

Tuần tự: các khối Conv2D, BatchNormalization, ReLU và MaxPooling2D được xếp nối tiếp.

Bản này giữ nguyên kiến trúc của baseline tương ứng; thay đổi cân bằng lớp nằm ở pipeline/huấn luyện.

In [3]:
def build_model(number_of_classes):
    model = models.Sequential(name="complex_cnn")
    model.add(layers.Input(shape=(*IMAGE_SIZE, 3)))

    model.add(layers.RandomFlip("horizontal"))
    model.add(layers.RandomRotation(15 / 360))
    model.add(layers.RandomTranslation(0.1, 0.1))
    model.add(layers.RandomZoom((-0.1, 0.1), width_factor=(-0.1, 0.1)))
    model.add(layers.RandomBrightness(0.2))
    model.add(layers.RandomContrast(0.2))
    model.add(layers.RandomSaturation(0.2))
    model.add(layers.Rescaling(1.0 / 255))

    # CNN block 1
    model.add(layers.Conv2D(32, (3, 3), padding="same", use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation("relu"))
    model.add(layers.MaxPooling2D((2, 2)))

    # CNN block 2
    model.add(layers.Conv2D(64, (3, 3), padding="same", use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation("relu"))
    model.add(layers.MaxPooling2D((2, 2)))

    # CNN block 3
    model.add(layers.Conv2D(128, (3, 3), padding="same", use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation("relu"))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.25))

    # CNN block 4
    model.add(layers.Conv2D(256, (3, 3), padding="same", use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation("relu"))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.3))

    # Classifier
    model.add(layers.GlobalAveragePooling2D())
    model.add(layers.Dense(128, activation="relu"))
    model.add(layers.Dropout(0.4))
    model.add(layers.Dense(number_of_classes, activation="softmax"))

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

## 3. Nạp và chuẩn bị dữ liệu

Ảnh được đọc theo cấu trúc `train/val/test/<lớp>`, resize đồng nhất và mã hóa nhãn one-hot cho `categorical_crossentropy`. Chuẩn hóa bằng `Rescaling(1/255)` đưa pixel về [0, 1].

Theo các phép augmentation của Model 1, chỉ trên dữ liệu train: lật ngang, xoay tối đa 15°, dịch chuyển tối đa 10% theo mỗi chiều, zoom khoảng 0.9–1.1, và biến đổi brightness/contrast/saturation với factor 0.2. Các lớp augmentation của Keras chỉ biến đổi ảnh khi `training=True`; validation và test không bị augmentation. Nếu hướng ảnh OCT có ý nghĩa lâm sàng thì cần cân nhắc bỏ lật ngang.

Validation dùng để theo dõi quá trình học; test được giữ riêng để đánh giá cuối.

In [4]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 64
EPOCHS = 40
EARLY_STOPPING_PATIENCE = 8
LR_REDUCTION_PATIENCE = 2
MIN_LEARNING_RATE = 0.00001

split_dirs = {split: data_dir / split for split in ("train", "val", "test")}
missing_dirs = [str(path) for path in split_dirs.values() if not path.is_dir()]
if missing_dirs:
    raise FileNotFoundError(f"Missing dataset split folders: {missing_dirs}")

train_ds = tf.keras.utils.image_dataset_from_directory(
    split_dirs["train"],
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    seed=SEED,
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    split_dirs["val"],
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False,
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    split_dirs["test"],
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False,
)

class_names = train_ds.class_names
if val_ds.class_names != class_names or test_ds.class_names != class_names:
    raise ValueError("Train, validation and test class folders must match.")

train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.prefetch(tf.data.AUTOTUNE)

image_extensions = {".bmp", ".gif", ".jpeg", ".jpg", ".png"}
class_counts = {
    class_name: sum(
        1 for path in (split_dirs["train"] / class_name).rglob("*")
        if path.is_file() and path.suffix.lower() in image_extensions
    )
    for class_name in class_names
}
if any(count == 0 for count in class_counts.values()):
    raise ValueError(f"Every class must have training images: {class_counts}")

number_of_train_images = sum(class_counts.values())
CLASS_WEIGHTS = {
    index: number_of_train_images / (len(class_names) * class_counts[class_name])
    for index, class_name in enumerate(class_names)
}
print("Classes:", class_names)
print("Training images by class:", class_counts)
print("Class weights:", CLASS_WEIGHTS)

Found 76515 files belonging to 4 classes.
Found 21861 files belonging to 4 classes.
Found 10933 files belonging to 4 classes.
Classes: ['CNV', 'DME', 'DRUSEN', 'NORMAL']
Training images by class: {'CNV': 26218, 'DME': 8118, 'DRUSEN': 6206, 'NORMAL': 35973}
Class weights: {0: 0.7296037073766115, 1: 2.356337767923134, 2: 3.082299387689333, 3: 0.5317529814027188}


## 4. Khởi tạo và kiểm tra mô hình

Tạo đầu ra theo số lớp lấy từ thư mục dữ liệu, sau đó kiểm tra kích thước batch ảnh, nhãn và đầu ra. Với bài toán này, đầu ra cần có 4 xác suất lớp; số tham số được in ra để hỗ trợ so sánh chi phí mô hình.

In [5]:
model = build_model(number_of_classes=len(class_names))
images, labels = next(iter(train_ds))
print("Input batch:", images.shape)
print("Label batch:", labels.shape)
print("Model output:", model(images, training=False).shape)
print(f"Trainable parameters: {model.count_params():,}")

Input batch: (64, 224, 224, 3)
Label batch: (64, 4)
Model output: (64, 4)
Trainable parameters: 423,268


## 5. Huấn luyện và lưu checkpoint tốt nhất

Mô hình cập nhật trọng số từ tập train và theo dõi `val_loss` trên validation. `ModelCheckpoint` giữ lại checkpoint có validation loss tốt nhất; `EarlyStopping` dừng khi không còn cải thiện để hạn chế học quá mức. Test không tham gia chọn checkpoint.

In [6]:
model_path = OUTPUT_DIR / f"{MODEL_NAME}_{DATASET}.keras"

train_started = time.perf_counter()
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    shuffle=False,
    class_weight=CLASS_WEIGHTS,
    callbacks=[
        tf.keras.callbacks.ModelCheckpoint(
            str(model_path),
            monitor="val_loss",
            save_best_only=True,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=LR_REDUCTION_PATIENCE,
            min_lr=MIN_LEARNING_RATE,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=EARLY_STOPPING_PATIENCE,
            restore_best_weights=True,
        ),
    ],
)

train_time = time.perf_counter() - train_started
print("Best checkpoint:", model_path)

Epoch 1/40


2026-10-10 00:01:26.650851: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/complex_cnn_1/dropout_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


1196/1196 ━━━━━━━━━━━━━━━━━━━━ 137s 111ms/step - accuracy: 0.6246 - loss: 1.1129 - val_accuracy: 0.7082 - val_loss: 0.7679 - learning_rate: 0.0010
Epoch 2/40
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 134s 112ms/step - accuracy: 0.7408 - loss: 0.7613 - val_accuracy: 0.7930 - val_loss: 0.5840 - learning_rate: 0.0010
Epoch 3/40
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 133s 111ms/step - accuracy: 0.8189 - loss: 0.5401 - val_accuracy: 0.8268 - val_loss: 0.4958 - learning_rate: 0.0010
Epoch 4/40
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 132s 110ms/step - accuracy: 0.8550 - loss: 0.4488 - val_accuracy: 0.8677 - val_loss: 0.3696 - learning_rate: 0.0010
Epoch 5/40
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 133s 111ms/step - accuracy: 0.8743 - loss: 0.3966 - val_accuracy: 0.9139 - val_loss: 0.2822 - learning_rate: 0.0010
Epoch 6/40
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 132s 111ms/step - accuracy: 0.8885 - loss: 0.3590 - val_accuracy: 0.8804 - val_loss: 0.3371 - learning_rate: 0.0010
Epoch 7/40
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 133s 112ms/step - acc

## 6. Đánh giá trên tập test

Sau khi chọn checkpoint bằng validation, đánh giá một lần trên test. Accuracy cho biết tỷ lệ dự đoán đúng tổng thể; precision, recall và F1-score theo từng lớp cùng confusion matrix giúp nhìn rõ các lớp bị nhầm.

In [7]:
best_model = tf.keras.models.load_model(model_path)
test_loss, test_accuracy = best_model.evaluate(test_ds, verbose=0)
y_true = np.concatenate([np.argmax(labels.numpy(), axis=1) for _, labels in test_ds])
y_pred = np.argmax(best_model.predict(test_ds, verbose=0), axis=1)

report = classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    zero_division=0,
    output_dict=True,
)
matrix = confusion_matrix(y_true, y_pred)
best_epoch = int(np.argmin(history.history["val_loss"]) + 1)
best_val_loss = float(np.min(history.history["val_loss"]))

print(f"Test loss: {test_loss:.4f} | Test accuracy: {test_accuracy:.4f}")
print("\nPrecision / Recall / F1-score:")
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))
print("Confusion matrix (rows=true, columns=predicted):")
print(matrix)

Test loss: 0.2024 | Test accuracy: 0.9293

Precision / Recall / F1-score:
              precision    recall  f1-score   support

         CNV       0.98      0.88      0.93      3746
         DME       0.85      0.91      0.88      1161
      DRUSEN       0.70      0.90      0.78       887
      NORMAL       0.97      0.97      0.97      5139

    accuracy                           0.93     10933
   macro avg       0.87      0.92      0.89     10933
weighted avg       0.94      0.93      0.93     10933

Confusion matrix (rows=true, columns=predicted):
[[3307  106  293   40]
 [  14 1061   14   72]
 [  30    6  798   53]
 [  21   82   42 4994]]


## 7. Lưu kết quả để so sánh

Lưu checkpoint Keras, metrics JSON, lịch sử CSV, confusion matrix dạng CSV và hai biểu đồ PNG (đường cong huấn luyện, ma trận nhầm lẫn) vào thư mục outputs của Model 2.

In [ ]:
results = {
    "model": f"Model 2 - {MODEL_NAME.replace('_', ' ').title()}",
    "dataset": DATASET,
    "params": int(model.count_params()),
    "img_size": IMAGE_SIZE[0],
    "epochs_run": len(history.history["loss"]),
    "train_time_min": round(train_time / 60, 2),
    "best_val_loss": best_val_loss,
    "test_loss": float(test_loss),
    "test_accuracy": float(test_accuracy),
    "test_macro_precision": float(report["macro avg"]["precision"]),
    "test_macro_recall": float(report["macro avg"]["recall"]),
    "test_macro_f1": float(report["macro avg"]["f1-score"]),
    "test_weighted_f1": float(report["weighted avg"]["f1-score"]),
    "classes": class_names,
    "classification_report": report,
}
with (OUTPUT_DIR / f"{MODEL_NAME}_{DATASET}_metrics.json").open("w", encoding="utf-8") as file:
    json.dump(results, file, ensure_ascii=False, indent=2)

history_rows = zip(
    history.history["loss"],
    history.history["val_loss"],
    history.history["accuracy"],
    history.history["val_accuracy"],
    history.history.get("learning_rate", [""] * len(history.history["loss"])),
)
with (OUTPUT_DIR / f"{MODEL_NAME}_{DATASET}_history.csv").open("w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["train_loss", "val_loss", "train_acc", "val_acc", "lr"])
    writer.writerows(history_rows)

with (OUTPUT_DIR / f"{MODEL_NAME}_{DATASET}_confusion_matrix.csv").open("w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["true/predicted", *class_names])
    writer.writerows([[name, *row] for name, row in zip(class_names, matrix)])

figure, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs = range(1, len(history.history["loss"]) + 1)
axes[0].plot(epochs, history.history["loss"], label="train")
axes[0].plot(epochs, history.history["val_loss"], label="validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[1].plot(epochs, history.history["accuracy"], label="train")
axes[1].plot(epochs, history.history["val_accuracy"], label="validation")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()
figure.tight_layout()
figure.savefig(OUTPUT_DIR / f"{MODEL_NAME}_{DATASET}_curves.png", dpi=150)
plt.show()
plt.close(figure)

figure, axis = plt.subplots(figsize=(6, 5))
image = axis.imshow(matrix, cmap="Blues")
axis.set(
    xticks=range(len(class_names)),
    yticks=range(len(class_names)),
    xticklabels=class_names,
    yticklabels=class_names,
    xlabel="Predicted label",
    ylabel="True label",
    title=f"{MODEL_NAME} - {DATASET} confusion matrix",
)
figure.colorbar(image, ax=axis)
threshold = matrix.max() / 2 if matrix.size else 0
for row in range(matrix.shape[0]):
    for column in range(matrix.shape[1]):
        axis.text(
            column,
            row,
            str(matrix[row, column]),
            ha="center",
            va="center",
            color="white" if matrix[row, column] > threshold else "black",
        )
figure.tight_layout()
figure.savefig(OUTPUT_DIR / f"{MODEL_NAME}_{DATASET}_confusion_matrix.png", dpi=150)
plt.show()
plt.close(figure)

print(f"Checkpoint: {model_path}")
print(f"Results saved in: {OUTPUT_DIR}")